# Icon sampler

Five small Icon programs, each its own section: Icon's *generators* and *goal-directed search* at work. Run All runs them in order; every cell can be run again on its own. The programs are from Rosetta Code and from the SCRIP demos, as they run in SCRIP.

## Eight queens, one solution

One procedure `q(c)` finds the row of the queen in column `c`. It is a generator: `suspend` offers each free row in turn and Icon backtracks to the next when a later column cannot be placed. The reversible assignment `<-` takes a square and gives it back on backtracking.

In [ ]:
procedure main()
    write(q(1), " ", q(2), " ", q(3), " ", q(4), " ", q(5), " ", q(6), " ", q(7), " ", q(8));
end

procedure q(c)
    static udiag, ddiag, row;

    initial {
        udiag := list(15, 0);
        ddiag := list(15, 0);
        row := list(8, 0)
    };

    every 0 = row[r := 1 to 8] = ddiag[r + c - 1] = udiag[8 + r - c] do   # test if free
        suspend row[r] <- ddiag[r + c - 1] <- udiag[8 + r - c] <- r    ;   # place and yield
end

## N queens, drawn

The same search as a board: `placequeen` offers each free square of a column, `solvequeen` goes on to the next column, and `show` draws the first solution on an 8 by 8 board.

In [ ]:
global n, rw, dd, ud


procedure solvequeen(c)
    if (c > n) then return show()
    else suspend placequeen(c) & solvequeen(c+1);
end

procedure placequeen(c)
    suspend (/rw[r := 1 to n] <- /dd[r+c-1] <- /ud[n+r-c] <- c);
end

procedure show()
    static count, line, border;
    initial {
        count := 0;
        line := repl("|   ",n) || "|";
        border := repl("----",n) || "-"
        };
    write("solution: ", count+:=1);
    write("  ", border);
    every line[4*(!rw - 1) + 3] <- "Q" do {
        write("  ", line);
        write("  ", border)
        };
    write();
    return;      # Comment out to see all possible solutions
end

In [ ]:
procedure main(args)
    n := integer(args[1]) | 8;
    rw := list(n);
    dd := list(2*n-1);
    ud := list(2*n-1);
    solvequeen(1);
end

## A knight's tour

Warnsdorff's rule: from each square go to the one with the fewest onward moves. The tour is printed as a board numbered by move.

In [ ]:
link printf


procedure KnightsTour(B,sq,tbrk,debug)  #: Warnsdorff’s algorithm

/B := Board(8);                          # create 8x8 board if none given
/sq := ?B.files || ?B.ranks;             # random initial position (default)
sq2fr(sq,B);                             # validate initial sq
if type(tbrk) == "procedure" then
   B.tiebreak := tbrk;                   # override tie-breaker
if \debug then write("Debug log : move#, move : (accessibility) choices");

choices := [];                           # setup to track moves and choices
every (movesto := table())[k := key(B.movesto)] := copy(B.movesto[k]);

B.tour := [];                            # new tour
repeat {
   put(B.tour,sq);                       # record move

   ac := 9;                              # accessibility counter > maximum
   while get(choices);                   # empty choices for tiebreak
   every delete(movesto[nextsq := !movesto[sq]],sq) do {  # make sq unavailable
      if ac >:= *movesto[nextsq] then   # reset to lower accessibility count
         while get(choices);             # . re-empty choices
      if ac = *movesto[nextsq] then
         put(choices,nextsq)            # keep least accessible sq and any ties
      };

   if \debug then {                     # move#, move, (accessibility), choices
      writes(sprintf("%d. %s : (%d) ",*B.tour,sq,ac));
      every writes(" ",!choices|"\n")
      };
   sq := B.tiebreak(choices,B) | break  # choose next sq until out of choices
   };
return B;
end

procedure RandomTieBreaker(S,B)                   # random choice
return ?S;
end

procedure FirstTieBreaker(S,B)                    # first one in the list
return !S;
end

procedure RothTieBreaker(S,B)                    # furthest from the center
if *S = 0 then fail;                              # must fail if []
every fr := sq2fr(s := !S,B) do {
   d := sqrt(abs(fr[1]-1 - (B.N-1)*0.5)^2 + abs(fr[2]-1 - (B.N-1)*0.5)^2);
   if (/md := d) | ( md >:= d) then msq := s     # save sq
   };
return msq;
end

record board(N,ranks,files,movesto,tiebreak,tour)  # structure for board

procedure Board(N)                      #: create board
N := *&lcase >=( 0 < integer(N)) | stop("N=",image(N)," is out of range.");
B := board(N,[],&lcase[1+:N],table(),RandomTieBreaker);       # setup
every put(B.ranks,N to 1 by -1) ;                             # add rank #s
every sq := !B.files || !B.ranks do                          # for each sq add
   every insert(B.movesto[sq] := set(), KnightMoves(sq,B));   # moves to next sq
return B;
end

procedure sq2fr(sq,B)                   #: return numeric file & rank
f := find(sq[1],B.files)                | runerr(205,sq);
r := integer(B.ranks[sq[2:0]])          | runerr(205,sq);
return [f,r];
end

procedure KnightMoves(sq,B)         #: generate all Kn accessible moves from sq
fr := sq2fr(sq,B);
every ( i := -2|-1|1|2 ) & ( j := -2|-1|1|2 ) do
   if (abs(i)~=abs(j)) & (0<(ri:=fr[2]+i)<=B.N) & (0<(fj:=fr[1]+j)<=B.N) then
      suspend B.files[fj]||B.ranks[ri];
end

procedure ShowTour(B)                   #: show the tour
write("Board size = ",B.N);
write("Tour length = ",*B.tour);
write("Tie Breaker = ",image(B.tiebreak));

every !(squares := list(B.N)) := list(B.N,"-");
every fr := sq2fr(B.tour[m := 1 to *B.tour],B) do
   squares[fr[2],fr[1]] := m;

every (hdr1 := "     ") ||:= right(!B.files,3);
every (hdr2 := "    +") ||:= repl((1 to B.N,"-"),3) | "-+";

every write(hdr1|hdr2);
every r := 1 to B.N do {
   writes(right(B.ranks[r],3)," |");
   every writes(right(squares[r,f := 1 to B.N],3));
   write(" |",right(B.ranks[r],3))
   };
every write(hdr2|hdr1|&null);
end

In [ ]:
procedure main(A)
ShowTour(KnightsTour(Board(8)));
end

## Most common words

String scanning: `?` scans a string, `upto` and `many` find the words, a table counts them and `sort(wt, 2)` orders the pairs by count. (The Rosetta program reads its text from standard input and reverses the sorted list with `reverse`, which Icon takes only for strings; here the text is one sentence and the loop counts down the sorted list.)

In [ ]:
procedure main()
   local WC, n, wt, L, i;
   WC := &letters ++ &digits;
   n := 5;
   wt := table(0);
   write("The ", n, " most common words are:");
   "the quick brown fox jumps over the lazy dog and the dog barks at the fox" ? while tab(upto(WC)) do wt[map(tab(many(WC)))] +:= 1;
   L := sort(wt, 2);
   every i := *L to *L - n + 1 by -1 do write(right(L[i][2], 8), ": ", L[i][1]);
end

## Every expression a grammar allows

A calculator's grammar in four procedures. The first cell walks it exhaustively, every expression of at most 2 tokens, the reversible assignment `<-` counting tokens on the path; the second walks it at random, one expression a call; the last cell runs both.

In [ ]:
global tokens, limit

procedure e_token(s)
    suspend (tokens <- tokens + 1, tokens <= limit, s);
end

procedure e_element()
    suspend ( e_token("x")
            | e_token("y")
            | e_token("z")
            | e_token("1")
            | (e_token("(") || e_expression() || e_token(")"))
            );
end

procedure e_factor()
    suspend ( e_element()
            | (e_token("+") || e_factor())
            | (e_token("-") || e_factor())
            );
end

procedure e_term()
    suspend ( e_factor()
            | (e_factor() || e_token("*") || e_term())
            | (e_factor() || e_token("/") || e_term())
            );
end

procedure e_expression()
    suspend ( e_term()
            | (e_term() || e_token("+") || e_expression())
            | (e_term() || e_token("-") || e_expression())
            );
end

In [ ]:
procedure r_element()
    local r;
    r := ?100;
    return ( (r <= 20, "x")
           | (r <= 40, "y")
           | (r <= 60, "z")
           | (r <= 80, string(?100))
           | ("(" || r_expression() || ")")
           );
end

procedure r_factor()
    local r;
    r := ?100;
    return ( (r <= 80, r_element())
           | (r <= 90, "+" || r_factor())
           | ("-" || r_factor())
           );
end

procedure r_term()
    local r;
    r := ?100;
    return ( (r <= 70, r_factor())
           | (r <= 85, r_factor() || "*" || r_term())
           | (r_factor() || "/" || r_term())
           );
end

procedure r_expression()
    local r;
    r := ?100;
    return ( (r <= 70, r_term())
           | (r <= 85, r_term() || "+" || r_expression())
           | (r_term() || "-" || r_expression())
           );
end

In [ ]:
procedure main()
   local n;
   limit := 2;
   tokens := 0;
   n := 0;
   every write(e_expression()) do n +:= 1;
   write(n, " expressions of at most ", limit, " tokens");
   &random := 1;
   every 1 to 8 do write(r_expression());
end